In [19]:
%matplotlib inline
import re, pickle, numpy as np, pandas as pd, matplotlib.pyplot as plt
from tensorflow.keras.models import model_from_json
from tensorflow.keras.preprocessing import sequence

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
%cd '/content/drive/My Drive/python github'

/content/drive/My Drive/python github


In [22]:
with open('imdb_model_architecture.json', 'r') as json_file:
    model = model_from_json(json_file.read())

model.load_weights('imdb_modell_weights.weights.h5')

In [23]:
model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

In [24]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 100, 32)        │        64,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 16)             │         3,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 67,153 (262.32 KB)

 Trainable params: 67,153 (262.32 KB)

 Non-trainable params: 0 (0.00 B)

In [25]:
with open('imdb_tokenizer.pickle', 'rb') as f:
    tokens = pickle.load(f)

In [26]:
#單筆影評測試
def rm_tags(text):
    return re.sub(r'<[^>]+>', '', text)

SentimentDict = {1: '正面的', 0: '負面的'}
#預測函數建立
def predict_review(input_text):
    cleaned_text = rm_tags(input_text)
    input_seq = tokens.texts_to_sequences([cleaned_text])
    pad_input_seq = sequence.pad_sequences(input_seq, maxlen=100)
    prob = float(model.predict(pad_input_seq, verbose=0)[0][0])
    return f"情意預測：{SentimentDict[round(prob)]}（正面評價機率：{prob:.4f}）"

In [27]:
# 影評 URL: https://letterboxd.com/film/godzilla-minus-zero/
url_1 = "https://letterboxd.com/film/godzilla-minus-zero/"
review_1 = '''
If minus one was peak, minus zero is good. The theme of standing up and making bigger than thou decisions was a welcome shift from Minus One’s perseverance. The score still slapped (I watched it in a Dolby enhanced theater though) tension was there in the airplane sequence early on. I felt the end kind of ended quickly, not as satisfying of a payoff as the last film. Also some unfinished plot lines I assume to address in the next film, Godzilla Plus One, Where Godzilla becomes a Dad
Also Godzilla doing a 911 was not on my bingo card…(technically would predate 911 but it was at New York so I mean…)
'''
# 影評 URL: https://letterboxd.com/apple__/film/dont-look-up-2021/
url_2 = "https://letterboxd.com/apple__/film/dont-look-up-2021/"
review_2 = '''
oscar-worthy performance by leo pretending to like women his own age
'''
# 影評 URL: https://letterboxd.com/kret123/film/the-hunger-games-sunrise-on-the-reaping/1/
url_3 = "https://letterboxd.com/kret123/film/the-hunger-games-sunrise-on-the-reaping/1/"
review_3 = '''
i want to apologize to suzanne collins for watching edits of young president snow. i will never touch them again
'''

In [28]:
reviews_list = [
    ("哥吉拉-0.0 - 正評", url_1, review_1),
    ("千萬別抬頭 - 負評", url_2, review_2),
    ("飢餓遊戲：鳴鳥與游蛇之歌 - 正評", url_3, review_3)
]

for idx, (title, url, text) in enumerate(reviews_list, 1):
    print(f"第 {idx} 則影評：{title}")
    print(f"影評 URL：{url}")
    print(predict_review(text))
    print("-" * 50)

第 1 則影評：哥吉拉-0.0 - 正評
影評 URL：https://letterboxd.com/film/godzilla-minus-zero/
情意預測：負面的（正面評價機率：0.0407）
--------------------------------------------------
第 2 則影評：千萬別抬頭 - 負評
影評 URL：https://letterboxd.com/apple__/film/dont-look-up-2021/
情意預測：正面的（正面評價機率：0.6453）
--------------------------------------------------
第 3 則影評：飢餓遊戲：鳴鳥與游蛇之歌 - 正評
影評 URL：https://letterboxd.com/kret123/film/the-hunger-games-sunrise-on-the-reaping/1/
情意預測：正面的（正面評價機率：0.5758）
--------------------------------------------------


In [29]:
!pip install -q gradio

In [30]:
import gradio as gr

In [31]:
iface = gr.Interface(
    fn=predict_review,
    inputs=gr.Textbox(
        lines=6,
        placeholder="請在此貼上英文 IMDb 影評文字...",
        label="輸入影評文字 (input_text)"
    ),
    outputs=gr.Textbox(label="模型情意預測結果"),
    title="IMDb 電影評論情意分析 Web App",
    description="輸入一段英文電影評論",
    examples=[
        [review_1.strip()],
        [review_2.strip()],
        [review_3.strip()]
    ]
)

iface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7c5f3ecd12ca234388.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
